In [1]:
# =========================================================
# IMPORTS
# =========================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T

from torchvision.datasets import CIFAR100
from torchvision.models import resnet18

from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
import random
import time
import os

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# =========================================================
# DEVICE CONFIG
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.backends.cudnn.benchmark = True

print("Device:", device)


# =========================================================
# EXPERIMENT PARAMETERS
# =========================================================

SSL_EPOCHS = 200
PROBE_EPOCHS = 100

BATCH_SIZE_SSL = 256
BATCH_SIZE_PROBE = 128

LR_SSL = 3e-4
LR_PROBE = 1e-3

JAC_WEIGHT = 0.01

SEEDS = [4]

OUTPUT_CSV = "cifar100_ssl_jacreg_results.csv"

print("SSL epochs:", SSL_EPOCHS)
print("Probe epochs:", PROBE_EPOCHS)

Torch: 2.9.0+cu126
CUDA available: True
Device: cuda
SSL epochs: 200
Probe epochs: 100


In [2]:
# =========================================================
# REPRODUCIBILITY
# =========================================================

def set_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

In [3]:
# =========================================================
# DATA AUGMENTATIONS
# =========================================================

class TwoCropsTransform:

    def __init__(self, base_transform):
        self.base_transform = base_transform

    def __call__(self, x):
        return self.base_transform(x), self.base_transform(x)


ssl_transform = TwoCropsTransform(

    T.Compose([
        T.RandomResizedCrop(32, scale=(0.2,1.0)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.4,0.4,0.4,0.1),
        T.RandomGrayscale(p=0.2),
        T.ToTensor(),
        T.Normalize((0.5071,0.4867,0.4408),
                    (0.2675,0.2565,0.2761))
    ])
)


probe_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.5071,0.4867,0.4408),
                (0.2675,0.2565,0.2761))
])

In [4]:
# =========================================================
# CIFAR100 DATASET
# =========================================================

ssl_dataset = CIFAR100(
    root="./data",
    train=True,
    download=True,
    transform=ssl_transform
)

probe_train = CIFAR100(
    root="./data",
    train=True,
    download=True,
    transform=probe_transform
)

probe_test = CIFAR100(
    root="./data",
    train=False,
    download=True,
    transform=probe_transform
)


ssl_loader = DataLoader(
    ssl_dataset,
    batch_size=BATCH_SIZE_SSL,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

probe_train_loader = DataLoader(
    probe_train,
    batch_size=BATCH_SIZE_PROBE,
    shuffle=True,
    num_workers=2
)

probe_test_loader = DataLoader(
    probe_test,
    batch_size=BATCH_SIZE_PROBE,
    shuffle=False,
    num_workers=2
)

print("Train size:", len(ssl_dataset))
print("Classes:", ssl_dataset.classes[:5], "... (100 total)")

100%|██████████| 169M/169M [00:01<00:00, 101MB/s]  


Train size: 50000
Classes: ['apple', 'aquarium_fish', 'baby', 'bear', 'beaver'] ... (100 total)


In [5]:
# =========================================================
# RESNET18 ENCODER
# =========================================================

class Encoder(nn.Module):

    def __init__(self):

        super().__init__()

        backbone = resnet18(weights=None)

        backbone.fc = nn.Identity()

        self.backbone = backbone

    def forward(self,x):

        return self.backbone(x)

In [6]:
# =========================================================
# PROJECTION HEAD
# =========================================================

class MLP(nn.Module):

    def __init__(self,in_dim=512,hidden_dim=2048,out_dim=128):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(in_dim,hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),

            nn.Linear(hidden_dim,out_dim)
        )

    def forward(self,x):

        return self.net(x)

In [7]:
# =========================================================
# SIMCLR MODEL
# =========================================================

class SimCLR(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = Encoder()

        self.projector = MLP()

    def forward(self,x):

        h = self.encoder(x)

        z = self.projector(h)

        return F.normalize(z,dim=1)

In [8]:
# =========================================================
# NT-XENT LOSS
# =========================================================

def nt_xent_loss(z1,z2,tau=0.5):

    N = z1.size(0)

    z = torch.cat([z1,z2],dim=0)

    sim = torch.mm(z,z.t()) / tau

    mask = torch.eye(2*N,device=z.device).bool()

    sim.masked_fill_(mask,-9e15)

    positives = torch.cat([
        torch.arange(N,2*N),
        torch.arange(0,N)
    ]).to(z.device)

    return F.cross_entropy(sim,positives)

In [9]:
# =========================================================
# JACOBIAN PENALTY
# =========================================================

def jacobian_penalty(model,x):

    x = x.clone().detach().requires_grad_(True)

    z = model(x)

    v = torch.randn_like(z)

    Jv = torch.autograd.grad(
        outputs=z,
        inputs=x,
        grad_outputs=v,
        retain_graph=True,
        create_graph=False
    )[0]

    return torch.sqrt((Jv**2).sum()/x.size(0))

In [10]:
# =========================================================
# METRICS
# =========================================================

def augmentation_sensitivity(encoder,x1,x2):

    with torch.no_grad():

        f1 = encoder(x1)

        f2 = encoder(x2)

    return (f1 - f2).norm(dim=1).mean().item()


def noise_sensitivity(encoder,x,noise_std=0.1):

    noise = noise_std * torch.randn_like(x)

    x_noisy = x + noise

    with torch.no_grad():

        f1 = encoder(x)

        f2 = encoder(x_noisy)

    return (f1 - f2).norm(dim=1).mean().item()

In [11]:
# =========================================================
# LINEAR PROBE
# =========================================================

class LinearProbe(nn.Module):

    def __init__(self,encoder):

        super().__init__()

        self.encoder = encoder

        for p in self.encoder.parameters():
            p.requires_grad=False

        self.fc = nn.Linear(512,100)

    def forward(self,x):

        with torch.no_grad():

            feat = self.encoder(x)

        return self.fc(feat)

In [12]:
# =========================================================
# PROBE TRAINING
# =========================================================

def train_probe(encoder):

    model = LinearProbe(encoder).to(device)

    optimizer = torch.optim.Adam(model.fc.parameters(),lr=LR_PROBE)

    criterion = nn.CrossEntropyLoss()

    for epoch in range(PROBE_EPOCHS):

        model.train()

        for x,y in probe_train_loader:

            x,y = x.to(device),y.to(device)

            loss = criterion(model(x),y)

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()


    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for x,y in probe_test_loader:

            x,y = x.to(device),y.to(device)

            pred = model(x).argmax(1)

            correct += (pred==y).sum().item()

            total += y.size(0)

    return correct/total

In [13]:
# =========================================================
# TRAIN SIMCLR
# =========================================================

def train_simclr(seed,use_jac=False):

    set_seed(seed)

    model = SimCLR().to(device)

    optimizer = torch.optim.Adam(model.parameters(),lr=LR_SSL)

    start = time.time()

    for epoch in range(SSL_EPOCHS):

        epoch_start = time.time()

        model.train()

        for (x1,x2),_ in ssl_loader:

            x1,x2 = x1.to(device),x2.to(device)

            z1 = model(x1)
            z2 = model(x2)

            loss = nt_xent_loss(z1,z2)

            if use_jac:

                jac = jacobian_penalty(model.encoder,x1)

                loss = loss + JAC_WEIGHT * jac

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

        print(f"Epoch {epoch+1}/{SSL_EPOCHS} | {time.time()-epoch_start:.1f}s")

    print("Training time (min):",(time.time()-start)/60)

    return model

In [14]:
# =========================================================
# MAIN EXPERIMENT
# =========================================================

results = []

for seed in SEEDS:

    print("\n==============================")
    print("RUNNING SEED:", seed)
    print("==============================")

    # -----------------------------
    # SimCLR
    # -----------------------------
    simclr_model = train_simclr(seed, use_jac=False)

    simclr_acc = train_probe(simclr_model.encoder)

    print(f"SimCLR Probe Accuracy: {simclr_acc:.4f}")


    # -----------------------------
    # SimCLR + Jacobian
    # -----------------------------
    simclr_jac_model = train_simclr(seed, use_jac=True)

    jac_acc = train_probe(simclr_jac_model.encoder)

    print(f"SimCLR+Jac Probe Accuracy: {jac_acc:.4f}")


    # -----------------------------
    # Save Results
    # -----------------------------
    results.append({
        "Seed": seed,
        "SimCLR_Probe_Acc": simclr_acc,
        "SimCLRJac_Probe_Acc": jac_acc
    })

    df = pd.DataFrame(results)

    # Save partial results (safety)
    df.to_csv(OUTPUT_CSV, index=False)

    print("\nCurrent Results Table")
    print(df)

    print("\nSaved to CSV:", OUTPUT_CSV)


    # -----------------------------
    # Cleanup GPU memory
    # -----------------------------
    del simclr_model
    del simclr_jac_model

    torch.cuda.empty_cache()


# =========================================================
# FINAL SUMMARY
# =========================================================

df = pd.DataFrame(results)

print("\n====================================")
print("FINAL RESULTS")
print("====================================")

print(df)


summary = df[["SimCLR_Probe_Acc","SimCLRJac_Probe_Acc"]].agg(["mean","std"])

print("\nMean ± Std Summary")
print(summary)


# Save final outputs
df.to_csv("final_cifar100_ssl_results.csv", index=False)
summary.to_csv("final_cifar100_ssl_summary.csv")

print("\nFiles saved:")
print("final_cifar100_ssl_results.csv")
print("final_cifar100_ssl_summary.csv")


RUNNING SEED: 4
Epoch 1/200 | 56.5s
Epoch 2/200 | 50.8s
Epoch 3/200 | 50.9s
Epoch 4/200 | 50.8s
Epoch 5/200 | 50.6s
Epoch 6/200 | 50.4s
Epoch 7/200 | 50.8s
Epoch 8/200 | 50.8s
Epoch 9/200 | 50.5s
Epoch 10/200 | 50.7s
Epoch 11/200 | 51.3s
Epoch 12/200 | 51.1s
Epoch 13/200 | 51.2s
Epoch 14/200 | 52.1s
Epoch 15/200 | 52.0s
Epoch 16/200 | 52.1s
Epoch 17/200 | 52.2s
Epoch 18/200 | 51.5s
Epoch 19/200 | 51.4s
Epoch 20/200 | 52.0s
Epoch 21/200 | 51.8s
Epoch 22/200 | 51.5s
Epoch 23/200 | 52.0s
Epoch 24/200 | 52.3s
Epoch 25/200 | 51.8s
Epoch 26/200 | 51.9s
Epoch 27/200 | 51.3s
Epoch 28/200 | 51.4s
Epoch 29/200 | 51.6s
Epoch 30/200 | 51.1s
Epoch 31/200 | 50.9s
Epoch 32/200 | 51.6s
Epoch 33/200 | 52.1s
Epoch 34/200 | 52.1s
Epoch 35/200 | 52.0s
Epoch 36/200 | 51.9s
Epoch 37/200 | 51.2s
Epoch 38/200 | 51.4s
Epoch 39/200 | 51.4s
Epoch 40/200 | 51.0s
Epoch 41/200 | 51.1s
Epoch 42/200 | 51.7s
Epoch 43/200 | 51.6s
Epoch 44/200 | 51.0s
Epoch 45/200 | 50.7s
Epoch 46/200 | 50.8s
Epoch 47/200 | 51.2s
Epoch